# 第五章 PyTorch 优化模块（Optimization）

> 适合 Google Colab：边运行、边理解、后续快速复习。  
> 原教程顺序：**5.1 损失函数 → 5.2 优化器 → 5.3 学习率调整器**

## 学习目标

学完本章，你应能解释：

1. loss 为什么通常接收 **logits**，以及 `reduction` 在做什么；
2. `loss.backward()`、`optimizer.step()`、`optimizer.zero_grad()` 各自负责什么；
3. Optimizer 如何通过 `param_groups` 管理参数，通过 `state` 保存动量等状态；
4. SGD 的一步更新如何与公式对应；
5. 为什么 Transformer / LLM 训练中经常使用 `AdamW`；
6. scheduler 如何修改 optimizer 中的学习率，以及 warmup + cosine 的基本结构。

## 版本说明（重要）

原教程第五章基于较早版本 PyTorch，用“21 个损失函数、13 个优化器、14 个学习率调整器”组织内容。**这些数量不是稳定知识，不建议背。**

本 Notebook 按当前 PyTorch 官方接口更新几个容易踩坑的点：

- `CrossEntropyLoss` 输入应直接给 **未归一化 logits**，不要先手动 `softmax`。
- `CrossEntropyLoss` 现在既支持类别索引 target，也支持类别概率 target；普通单标签分类优先用类别索引。
- `Optimizer.zero_grad()` 当前默认 `set_to_none=True`，梯度会被置为 `None`，而不是一定变成全 0 Tensor。
- 学习率调度器的公开基类使用 `torch.optim.lr_scheduler.LRScheduler`。
- `scheduler.step()` 通常放在 `optimizer.step()` 之后；`OneCycleLR` 等按 batch 更新的调度器需要每个 batch 调一次。

教程：
- https://tingsongyu.github.io/PyTorch-Tutorial-2nd/chapter-5/
- https://tingsongyu.github.io/PyTorch-Tutorial-2nd/chapter-5/5.1-loss-function.html
- https://tingsongyu.github.io/PyTorch-Tutorial-2nd/chapter-5/5.2-Optimizer.html
- https://tingsongyu.github.io/PyTorch-Tutorial-2nd/chapter-5/5.3-lr-scheduler.html

官方文档：
- https://docs.pytorch.org/docs/stable/generated/torch.nn.CrossEntropyLoss.html
- https://docs.pytorch.org/docs/stable/optim.html
- https://docs.pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate


In [ ]:
import copy
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

# 5.1 损失函数（Loss Function）

损失函数把“预测有多差”压缩成可优化的数值。训练的核心链路是：

$$
\text{logits} \rightarrow \text{loss} \rightarrow \text{backward} \rightarrow \nabla_\theta L
$$

原教程列出的 21 个损失函数包括：

`L1Loss`、`MSELoss`、`CrossEntropyLoss`、`CTCLoss`、`NLLLoss`、`PoissonNLLLoss`、`GaussianNLLLoss`、`KLDivLoss`、`BCELoss`、`BCEWithLogitsLoss`、`MarginRankingLoss`、`HingeEmbeddingLoss`、`MultiLabelMarginLoss`、`HuberLoss`、`SmoothL1Loss`、`SoftMarginLoss`、`MultiLabelSoftMarginLoss`、`CosineEmbeddingLoss`、`MultiMarginLoss`、`TripletMarginLoss`、`TripletMarginWithDistanceLoss`。

不需要逐个记忆。先掌握教程重点：**L1Loss 的实现机制 + CrossEntropyLoss 的正确输入形式**。

## 5.1.1 L1Loss：先理解 `reduction`

逐元素 L1 loss：

$$
l_i = |x_i-y_i|
$$

`reduction` 决定最后如何汇总：

- `"none"`：保留每个元素的 loss；
- `"mean"`：取平均，最常用；
- `"sum"`：求和。

`size_average`、`reduce` 属于旧接口，现代代码只使用 `reduction`。

In [ ]:
pred = torch.tensor([1.0, 3.0, -1.0])
target = torch.tensor([0.0, 1.0, 1.0])

for reduction in ["none", "mean", "sum"]:
    loss = nn.L1Loss(reduction=reduction)(pred, target)
    print(f"{reduction:>4} -> {loss}")

# 手算验证：|1-0|, |3-1|, |-1-1| = [1, 2, 2]
print("manual:", (pred - target).abs())

## 5.1.2 PyTorch Loss 的实现层次

原教程重点分析的结构可以抽象为：

```text
nn.L1Loss (Module)
    ↓ forward()
torch.nn.functional.l1_loss
    ↓
底层 ATen / C++ kernel
```

对学习者最重要的是：**Loss 类本质也是 `nn.Module`**。你自定义 loss 时，通常只需实现 `forward()`；底层高性能算子由 PyTorch 完成。

In [ ]:
class MyL1Loss(nn.Module):
    def __init__(self, reduction="mean"):
        super().__init__()
        self.reduction = reduction

    def forward(self, input, target):
        # 直接委托给 functional API，保持 autograd 链路
        return F.l1_loss(input, target, reduction=self.reduction)

x = torch.tensor([1.0, 2.0], requires_grad=True)
y = torch.tensor([0.0, 4.0])

loss_builtin = nn.L1Loss()(x, y)
loss_custom = MyL1Loss()(x, y)

print("same:", torch.allclose(loss_builtin, loss_custom))
loss_custom.backward()
print("x.grad:", x.grad)

## 5.1.3 CrossEntropyLoss：分类最重要的 loss

对于类别索引 target，核心关系可理解为：

$$
\operatorname{CrossEntropy}(z,y)
=
-\log \operatorname{softmax}(z)_y
$$

其中 $z$ 是模型输出的 **logits**。

### 三个必须记住的点

1. `input` 直接给 logits，**不要提前 softmax**；
2. 普通单标签分类中，target 通常是 `torch.long`，shape 为 `[N]`；
3. logits 常见 shape 为 `[N, C]`，其中 `C` 是类别数。

当前 PyTorch 也允许 target 是 `[N, C]` 的类别概率（如 mixup / soft labels），因此原教程“target 必须是 int、不能 one-hot”的说法只适用于最常见的**类别索引模式**。

In [ ]:
logits = torch.tensor([
    [2.0, 0.5, -1.0],
    [0.1, 1.2, 0.3],
], requires_grad=True)

target = torch.tensor([0, 1], dtype=torch.long)

ce = F.cross_entropy(logits, target)
manual = F.nll_loss(F.log_softmax(logits, dim=1), target)

print("logits shape:", logits.shape)
print("target shape:", target.shape)
print("CE:", ce.item())
print("LogSoftmax + NLLLoss:", manual.item())
print("same:", torch.allclose(ce, manual))

ce.backward()
print("gradient shape:", logits.grad.shape)
print("gradient:\n", logits.grad)

## 5.1.4 二分类：优先 `BCEWithLogitsLoss`

二分类 / 多标签任务中，常见写法是：

```text
logits → BCEWithLogitsLoss
```

而不是：

```text
logits → sigmoid → BCELoss
```

两者数学上对应，但 `BCEWithLogitsLoss` 把 sigmoid 与 BCE 合并，数值稳定性更好。

In [ ]:
binary_logits = torch.tensor([2.0, -1.0, 0.3], requires_grad=True)
binary_target = torch.tensor([1.0, 0.0, 1.0])

loss_stable = F.binary_cross_entropy_with_logits(binary_logits, binary_target)
loss_two_steps = F.binary_cross_entropy(binary_logits.sigmoid(), binary_target)

print("BCEWithLogits:", loss_stable.item())
print("sigmoid + BCE:", loss_two_steps.item())
print("close:", torch.allclose(loss_stable, loss_two_steps, atol=1e-6))

### 5.1 小结：如何选 loss

| 任务 | 常用 loss | 模型最后一层通常输出 |
|---|---|---|
| 回归 | `MSELoss` / `L1Loss` / `HuberLoss` | 实数 |
| 单标签多分类 | `CrossEntropyLoss` | logits，不做 softmax |
| 二分类 / 多标签 | `BCEWithLogitsLoss` | logits，不做 sigmoid |
| 语言模型 next-token prediction | `CrossEntropyLoss` | `[batch, seq, vocab]` logits |

**LLM 连接点：**自回归语言模型本质上是在每个 token 位置做词表上的多分类，因此训练核心 loss 通常仍是交叉熵。

# 5.2 优化器（Optimizer）

教程把 Optimizer 的工作拆成三个问题：

1. **梯度从哪里来？** `loss.backward()` 由 autograd 计算并写入 `param.grad`；
2. **更新哪些参数？** optimizer 只管理初始化时传入的参数；
3. **如何更新？** `optimizer.step()` 根据具体优化算法修改参数。

最小训练顺序：

```python
optimizer.zero_grad()
loss = loss_fn(model(x), y)
loss.backward()
optimizer.step()
```

原教程列出的 13 个优化器包括：`SGD`、`Adadelta`、`Adagrad`、`Adam`、`AdamW`、`SparseAdam`、`Adamax`、`ASGD`、`LBFGS`、`NAdam`、`RAdam`、`RMSprop`、`Rprop`。

当前 PyTorch 已继续增加新优化器，所以**不要把“13 个”当固定事实**。

In [ ]:
# 用一个参数观察 backward -> grad -> step
w = nn.Parameter(torch.tensor([1.0]))
optimizer = torch.optim.SGD([w], lr=0.1)

optimizer.zero_grad()
loss = (w - 3.0).pow(2).sum()
loss.backward()

print("before step: w =", w.item(), "grad =", w.grad.item())
optimizer.step()
print("after step:  w =", w.item())

# 梯度为 d(w-3)^2/dw = 2(w-3) = -4
# SGD: w_new = 1 - 0.1 * (-4) = 1.4

## 5.2.1 Optimizer 的核心数据结构

教程重点讲了三个属性：

- `param_groups`：optimizer 管理的参数分组，以及每组的 `lr`、`weight_decay` 等超参数；
- `state`：与参数绑定的优化状态，例如 momentum buffer、Adam 的一阶/二阶矩；
- `defaults`：optimizer 的默认超参数。

以及常用方法：

- `zero_grad()`：清理上一轮梯度；
- `step()`：更新参数；
- `add_param_group()`：增加一组参数；
- `state_dict()` / `load_state_dict()`：保存与恢复优化器状态。

In [ ]:
model = nn.Linear(4, 2)
optimizer = torch.optim.SGD(model.parameters(), lr=0.1, momentum=0.9)

x = torch.randn(3, 4)
y = torch.randn(3, 2)

optimizer.zero_grad()
F.mse_loss(model(x), y).backward()
optimizer.step()  # 执行一次后，momentum state 才真正建立

print("number of param groups:", len(optimizer.param_groups))
print("group keys:", sorted(k for k in optimizer.param_groups[0].keys() if k != "params"))
print("state entries:", len(optimizer.state))
print("defaults:", optimizer.defaults)

## 5.2.2 为什么必须清梯度？

PyTorch 默认会把多次 `backward()` 的梯度**累加**到 `.grad`。

这既可能是 bug，也可以被有意用于 **gradient accumulation（梯度累积）**。

当前 `optimizer.zero_grad()` 默认 `set_to_none=True`：

- 更省内存；
- 没收到梯度的参数会保持 `.grad is None`；
- 如果你的代码必须手工读取 `.grad`，要注意 `None` 与全 0 Tensor 的区别。

In [ ]:
p = nn.Parameter(torch.tensor([2.0]))

(p ** 2).backward()
print("after first backward:", p.grad.item())   # 4

(p ** 2).backward()
print("after second backward:", p.grad.item())  # 累加为 8

p.grad = None
(p ** 2).backward()
print("after clear:", p.grad.item())            # 重新变为 4

## 5.2.3 参数组：不同参数使用不同超参数

`param_groups` 对微调尤其重要。例如：

- backbone 用小学习率；
- 新增分类头用大学习率；
- 对 bias / norm 参数设置不同 weight decay；
- 冻结层解冻后，通过 `add_param_group()` 加入 optimizer。

Transformer / LLM 工程中，也常按“需要 weight decay / 不需要 weight decay”对参数分组。

In [ ]:
model = nn.Sequential(
    nn.Linear(4, 8),  # 假设是预训练 backbone
    nn.ReLU(),
    nn.Linear(8, 2),  # 假设是新任务 head
)

optimizer = torch.optim.SGD([
    {"params": model[0].parameters(), "lr": 1e-3},
    {"params": model[2].parameters(), "lr": 1e-2},
], momentum=0.9)

for i, group in enumerate(optimizer.param_groups):
    print(f"group {i}: lr={group['lr']}, num_params={len(group['params'])}")

## 5.2.4 SGD：把代码与公式对应起来

最基础的 SGD：

$$
w_{t+1}=w_t-\eta g_t
$$

其中 $\eta$ 是学习率，$g_t=\nabla_w L$。

在 SGD 中加入 `weight_decay=\lambda` 时，可理解为：

$$
w_{t+1}=w_t-\eta(g_t+\lambda w_t)
$$

再加入 momentum 后，optimizer 的 `state` 中就需要保存历史动量。

In [ ]:
w = nn.Parameter(torch.tensor([2.0]))
optimizer = torch.optim.SGD([w], lr=0.1, weight_decay=0.01)

optimizer.zero_grad()
loss = (w - 3.0).pow(2).sum()
loss.backward()

old_w = w.detach().clone()
grad = w.grad.detach().clone()
expected = old_w - 0.1 * (grad + 0.01 * old_w)

optimizer.step()

print("manual expected:", expected.item())
print("PyTorch result :", w.item())
print("same:", torch.allclose(w.detach(), expected))

## 5.2.5 AdamW：与 LLM 最相关的优化器

`Adam` 会为每个参数维护梯度的一阶矩、二阶矩；`AdamW` 则把 **weight decay 与梯度更新解耦（decoupled weight decay）**。

对 Transformer / LLM，你应优先理解：

- 为什么 Adam/AdamW 需要额外 optimizer state；
- `betas=(β1, β2)` 在追踪一阶/二阶统计；
- weight decay 与普通 L2 正则并不总是等价；
- 实际训练通常还会配合 warmup + decay scheduler。

这里不推导完整 AdamW，只观察它的 state。

In [ ]:
model = nn.Linear(4, 2)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=0.01)

x = torch.randn(8, 4)
target = torch.randn(8, 2)

optimizer.zero_grad()
loss = F.mse_loss(model(x), target)
loss.backward()
optimizer.step()

first_param = next(model.parameters())
state = optimizer.state[first_param]

print("AdamW state keys:", state.keys())
print("step:", state["step"])
print("exp_avg shape:", state["exp_avg"].shape)
print("exp_avg_sq shape:", state["exp_avg_sq"].shape)

### 5.2 小结

训练时真正需要熟练的是这条链路：

```text
optimizer.zero_grad()
        ↓
forward → loss
        ↓
loss.backward()      # 产生 / 累积 .grad
        ↓
optimizer.step()     # 读取 .grad，修改 Parameter
```

恢复训练时，**模型参数 + optimizer state + scheduler state** 通常都要保存；只保存模型权重不能完整恢复动量、Adam 统计量与当前学习率进度。

# 5.3 学习率调整器（LR Scheduler）

Scheduler 不直接管理模型参数，它管理的是 **optimizer 中各 parameter group 的学习率**：

```text
scheduler
   ↓ 修改
optimizer.param_groups[i]["lr"]
   ↓
影响下一次 optimizer.step()
```

原教程基于 `_LRScheduler` 讲内部设计；当前公开基类为 `torch.optim.lr_scheduler.LRScheduler`。

常见状态 / 接口：

- `optimizer`：被管理的优化器；
- `base_lrs`：初始学习率；
- `last_epoch`：调度进度；
- `step()`：推进一次调度；
- `get_last_lr()`：查看最近一次计算出的学习率；
- `state_dict()` / `load_state_dict()`：保存 / 恢复调度器状态。

## 5.3.1 StepLR：最适合看懂 scheduler 机制

`StepLR(step_size=k, gamma=γ)` 每隔 `k` 次 scheduler step：

$$
\eta_{\text{new}}=\gamma \eta_{\text{old}}
$$

普通 epoch-based scheduler 的典型顺序：

```python
for epoch in ...:
    train(...)
    optimizer.step()
    scheduler.step()
```

注意：`OneCycleLR` 这类 scheduler 是 **每个 batch** 更新，不应照搬 epoch-based 调用频率。

In [ ]:
p = nn.Parameter(torch.tensor([0.0]))
optimizer = torch.optim.SGD([p], lr=0.1)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=2, gamma=0.5)

used_lrs = []
for epoch in range(6):
    optimizer.zero_grad()
    (p * 0).sum().backward()  # 仅构造一个最小可运行优化步骤
    optimizer.step()

    used_lrs.append(optimizer.param_groups[0]["lr"])
    scheduler.step()

print("lr used in each epoch:", used_lrs)
print("scheduler last lr:", scheduler.get_last_lr())

## 5.3.2 原教程的 14 类 scheduler：理解“策略”，不要背数量

| 类型 | 代表 API | 直觉 |
|---|---|---|
| 函数式 | `LambdaLR`, `MultiplicativeLR` | 自定义变化规则 |
| 分段衰减 | `StepLR`, `MultiStepLR` | 到固定阶段再降 |
| 线性/常量 | `ConstantLR`, `LinearLR` | 常用于 warmup |
| 指数 | `ExponentialLR` | 每步乘固定比例 |
| 余弦 | `CosineAnnealingLR` | 平滑下降，Transformer/LLM 常见 |
| 组合 | `ChainedScheduler`, `SequentialLR` | 把多个阶段串起来 |
| 看验证指标 | `ReduceLROnPlateau` | 指标不再改善时降 lr |
| 周期型 | `CyclicLR`, `OneCycleLR` | batch 级动态学习率 |
| 余弦重启 | `CosineAnnealingWarmRestarts` | 周期性重新升高 lr |

当前 PyTorch 还提供了更多策略（例如 `PolynomialLR`），因此 API 数量会随版本变化。

## 5.3.3 LLM 常见模式：warmup + cosine decay

训练初期直接使用较大学习率可能不稳定，因此常先 warmup：

$$
\eta_t \uparrow
$$

再进行 cosine decay：

$$
\eta_t \downarrow
$$

PyTorch 可以用 `LinearLR + CosineAnnealingLR + SequentialLR` 表达这个结构。

In [ ]:
p = nn.Parameter(torch.tensor([0.0]))
optimizer = torch.optim.AdamW([p], lr=1e-3)

warmup = torch.optim.lr_scheduler.LinearLR(
    optimizer,
    start_factor=0.1,
    end_factor=1.0,
    total_iters=2,
)
cosine = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=6,
    eta_min=1e-5,
)
scheduler = torch.optim.lr_scheduler.SequentialLR(
    optimizer,
    schedulers=[warmup, cosine],
    milestones=[2],
)

lrs = []
for step in range(8):
    optimizer.zero_grad()
    (p * 0).sum().backward()
    optimizer.step()

    lrs.append(optimizer.param_groups[0]["lr"])
    scheduler.step()

print("warmup + cosine lr:")
for i, lr in enumerate(lrs):
    print(f"step {i}: {lr:.8f}")

## 5.3.4 三个模块放在一起：完整最小训练循环

下面用纯模拟数据完成：

```text
数据 → 模型 → CrossEntropyLoss → backward
    → AdamW.step() → cosine scheduler.step()
```

这就是后续训练 CNN、Transformer、LLM 时会不断扩展的基本骨架。

In [ ]:
torch.manual_seed(42)

# 模拟 3 分类数据，不依赖外部文件
x = torch.randn(256, 8)
teacher = torch.randn(8, 3)
y = (x @ teacher).argmax(dim=1)

model = nn.Sequential(
    nn.Linear(8, 32),
    nn.ReLU(),
    nn.Linear(32, 3),
)

optimizer = torch.optim.AdamW(model.parameters(), lr=3e-2, weight_decay=1e-2)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=20, eta_min=1e-3
)

history = []

for epoch in range(20):
    optimizer.zero_grad()

    logits = model(x)
    loss = F.cross_entropy(logits, y)

    loss.backward()
    optimizer.step()
    scheduler.step()

    with torch.no_grad():
        acc = (logits.argmax(dim=1) == y).float().mean().item()

    history.append((loss.item(), acc, optimizer.param_groups[0]["lr"]))

print("epoch  1:", history[0])
print("epoch 20:", history[-1])

# 本章知识结构总结

```text
Loss
├─ 衡量预测与目标的差异
├─ Module.forward → functional → 底层 kernel
├─ reduction: none / mean / sum
└─ LLM: CrossEntropyLoss(logits, token_id)

Optimizer
├─ backward() 产生 param.grad
├─ param_groups 决定管理哪些参数、各组超参数
├─ state 保存 momentum / Adam moments
├─ zero_grad() 清理梯度
└─ step() 真正更新 Parameter

LR Scheduler
├─ 不直接改模型参数
├─ 修改 optimizer.param_groups[*]["lr"]
├─ epoch-based / batch-based 调用频率不同
└─ LLM 常见：warmup → cosine decay
```

## 后续学习 Transformer / LLM 时重点保留

1. **CrossEntropyLoss 直接接 logits**；
2. token prediction 本质是大规模多分类；
3. `AdamW` 的 state 会占用额外显存；
4. 参数组可实现不同 weight decay / learning rate；
5. warmup + cosine 是非常常见的训练调度结构；
6. gradient accumulation 利用了“梯度默认累加”这一机制。

# 学完必须会回答的问题

1. `CrossEntropyLoss` 为什么不应该先对 logits 手动 `softmax`？
2. `reduction="none" / "mean" / "sum"` 分别意味着什么？
3. `nn.L1Loss` 为什么可以像函数一样写成 `loss_fn(input, target)`？
4. `loss.backward()` 和 `optimizer.step()` 的职责有什么本质区别？
5. 为什么连续两次 `backward()` 不清梯度会发生梯度累加？
6. `optimizer.param_groups` 解决了什么问题？微调时为什么有用？
7. optimizer 的 `state` 里通常保存什么？为什么 AdamW 比 SGD 需要更多状态？
8. 写出最基础 SGD 更新公式，并解释学习率的作用。
9. SGD 中的 `weight_decay` 如何进入更新公式？它和 AdamW 的 decoupled weight decay 有何概念差别？
10. `scheduler.step()` 实际修改的是哪里？
11. 为什么 `OneCycleLR` 不能简单按每个 epoch 调一次？
12. 为什么 Transformer / LLM 训练常见 `AdamW + warmup + cosine decay`？
